In [1]:
import pandas as pd

In [2]:
stk_data=pd.read_csv("TATACOFEE_1321.csv")

In [3]:
stk_data

,Date,Price,Open,High,Low,Vol.,Change %
0,15-01-2013,"1,586.95","1,605.45","1,605.45","1,582.20",57.64K,-0.42%
1,16-01-2013,"1,572.70","1,588.05","1,591.15","1,570.00",62.35K,-0.90%
2,17-01-2013,"1,599.90","1,577.15","1,645.80","1,570.00",225.98K,1.73%
3,18-01-2013,"1,587.00","1,609.80","1,614.70","1,578.55",71.61K,-0.81%
4,21-01-2013,"1,596.90","1,594.75","1,621.00","1,587.95",64.27K,0.62%
...,...,...,...,...,...,...,...
2210,27-12-2021,218.35,200.00,222.00,196.00,5.89M,8.63%
2211,28-12-2021,212.35,219.65,220.45,211.55,2.87M,-2.75%
2212,29-12-2021,211.35,213.00,216.70,210.00,2.71M,-0.47%
2213,30-12-2021,208.50,211.45,211.50,207.90,977.48K,-1.35%


In [4]:
stk_data.rename(columns={"Price":"Close"},inplace=True)

In [5]:
import warnings
warnings.filterwarnings("ignore")

In [6]:
# Filtering data for particular dates
stk_data["Date"]=pd.to_datetime(stk_data["Date"])
stk_data=stk_data[(stk_data["Date"]>="2020-07-01")&(stk_data["Date"]<="2021-12-31")]

In [7]:
stk_data=stk_data.sort_values("Date")
stk_data=stk_data.set_index("Date")

In [8]:
stk_data

,Close,Open,High,Low,Vol.,Change %
Date,,,,,,
2020-07-01,81.95,81.50,82.70,81.05,318.89K,0.00%
2020-07-02,81.90,82.25,82.70,81.55,267.38K,-0.06%
2020-07-03,84.35,82.05,85.45,82.05,921.18K,2.99%
2020-07-06,86.60,85.80,87.85,85.05,1.66M,2.67%
2020-07-07,85.65,86.70,87.00,85.05,476.14K,-1.10%
...,...,...,...,...,...,...
2021-12-27,218.35,200.00,222.00,196.00,5.89M,8.63%
2021-12-28,212.35,219.65,220.45,211.55,2.87M,-2.75%
2021-12-29,211.35,213.00,216.70,210.00,2.71M,-0.47%


In [9]:
# As we have values with K and M in Volume(i.e string) we need to convert them into numbers
# writing a function for the same
def convert_volume(x):
    if isinstance(x,str):
      x=x.strip()
      if x.endswith('K'):
          return(float(x[:-1])*1000)
      elif x.endswith('M'):
          return(float(x[:-1])*100000)
    return float(X)

In [10]:
# using the above function to convert the volume values in the data
stk_data["Vol."]=stk_data["Vol."].apply(convert_volume)

In [11]:
stk_data["Vol."]

Date
2020-07-01    318890.0
2020-07-02    267380.0
2020-07-03    921180.0
2020-07-06    166000.0
2020-07-07    476140.0
                ...   
2021-12-27    589000.0
2021-12-28    287000.0
2021-12-29    271000.0
2021-12-30    977480.0
2021-12-31    305000.0
Name: Vol., Length: 376, dtype: float64

In [12]:
# preprocessing
from sklearn.preprocessing import MinMaxScaler
Ms=MinMaxScaler()
cols=["Open","High","Low","Close","Vol."]
data1=pd.DataFrame(
    Ms.fit_transform(stk_data[cols]),
    columns=cols,
    index=stk_data.index
)
print("Len:",data1.shape)

Len: (376, 5)


In [13]:
data1 = data1.reset_index(drop=True)

In [14]:
print(type(data1.index))
print(data1.index[:5])




<class 'pandas.core.indexes.range.RangeIndex'>
RangeIndex(start=0, stop=5, step=1)


In [15]:
# pip install statsmodels

In [16]:
performance = {
    "Model": [],
    "RMSE": [],
    "MaPe": [],
    "Order": [],
    "Test": []
}

In [17]:
from statsmodels.tsa.statespace.varmax import VARMAX
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_percentage_error


In [18]:
def varmax_model(dataset,listt):
    print(listt)
    test_obs=28
    # endongenous variables
    datasetTwo=dataset[listt]
    # exogenous variable
    exog=dataset[["Vol."]]
    # Train and Test split
    train=datasetTwo[:-test_obs]
    test=datasetTwo[-test_obs:]
    exog_train=exog[:-test_obs]
    exog_test=exog[-test_obs:]
    # Determining AIC values for different ('p,q) combinations
    aic_values=[]
    for p in [1,2]:
        for q in [0,1]:
            model=VARMAX(
                train,
                exog=exog_train,
                order=(p,q)
            )
        results=model.fit(disp=False)
        print("Order:",(p,q))
        print("Aic:",results.aic)
        # inserting a blank line so the next order and Aic values are readable on screen
        print()
        # here we are storing order and aic value of that order in aic_values
        aic_values.append(((p,q),results.aic))
        # Select the order with minimum aic value
        # Here we are using lambda for extracting the aic value alone for the aic_values as if it has both order and aic value, so x[1] 
        # used to extract aic value from the x(i.e aic_values)
        #[0] is used to take the order which has minimum aic values
        order=min(aic_values, key=lambda x:x[1])[0]
        print("Selected order=",order)
        # fit final model
        model=VARMAX(
               train,
               exog=exog_train,
               order=order
        )
        result=model.fit(disp=False)
        # forecasting the next series
        pred=result.forecast(
                   steps=28,
                   exog=exog_test
        )
        
        print("prediction")
        print(pred)
        # RMSE
        rmse=round(mean_squared_error(test,pred)**0.5,4)
        #MAPE
        mape=mean_absolute_percentage_error(test,pred)
        print("RMSE:",rmse)
        print("MAPE:",mape)
        performance["Model"].append(listt)
        performance["RMSE"].append(rmse)
        performance["MaPe"].append(mape)
        performance["Order"].append(order)
        performance["Test"].append(test_obs)
        print({key: len(value) for key, value in performance.items()})
        perf=pd.DataFrame(performance)
        return pred,result,perf

In [19]:
import warnings
warnings.filterwarnings("ignore")

In [20]:
# TO get all the combinations in one function the below code can be used
listt = [
    ['Close', 'High'],
    ['Close', 'High', 'Open'],
    ['Close', 'High', 'Open', 'Low']
]

for combination in listt:
    pred,result,perf = varmax_model(data1, combination)

['Close', 'High']
Order: (1, 1)
Aic: -3597.343648642232

Selected order= (1, 1)
prediction
        Close      High
348  0.882378  0.811450
349  0.880324  0.818021
350  0.877857  0.810609
351  0.875463  0.808068
352  0.873039  0.805003
353  0.871074  0.810721
354  0.868909  0.807099
355  0.866915  0.807579
356  0.864792  0.803815
357  0.862415  0.796506
358  0.860097  0.793797
359  0.858208  0.799164
360  0.855901  0.791480
361  0.853970  0.794533
362  0.851920  0.791703
363  0.849862  0.789202
364  0.847879  0.788323
365  0.845891  0.786487
366  0.843615  0.779155
367  0.841356  0.775756
368  0.839116  0.773460
369  0.837167  0.776449
370  0.835245  0.776258
371  0.833247  0.773335
372  0.831105  0.768365
373  0.828984  0.765775
374  0.827256  0.770648
375  0.825136  0.763445
RMSE: 0.0498
MAPE: 0.05251426369092923
{'Model': 1, 'RMSE': 1, 'MaPe': 1, 'Order': 1, 'Test': 1}
['Close', 'High', 'Open']
Order: (1, 1)
Aic: -5679.893745260598

Selected order= (1, 1)
prediction
        Close    

In [21]:
perf

,Model,RMSE,MaPe,Order,Test
0,"[Close, High]",0.0498,0.052514,"(1, 1)",28
1,"[Close, High, Open]",0.0518,0.053636,"(1, 1)",28
2,"[Close, High, Open, Low]",0.0452,0.047479,"(1, 1)",28


In [22]:
perf.to_csv("varmax_result.csv",index=False)